In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 🧪 Competição KAGGLE




In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/processado/kaggle.zip' '/content/kaggle.zip'

In [ ]:
!unzip "/content/kaggle.zip"  -d /..

A saída de streaming foi truncada nas últimas 5000 linhas.
  inflating: /../content/kaggle/carSide/0F8A8DM24V8S9ORGBRR3_3.png  
  inflating: /../content/kaggle/carSide/8PORD6BGOTB9DZ9F4E8E_5.png  
  inflating: /../content/kaggle/carSide/LG8CHSF71R8O2UPWJWLW_0.png  
  inflating: /../content/kaggle/carSide/6AWLWN8UOB3BIGMMX2GK_1.png  
  inflating: /../content/kaggle/carSide/HHKPGT9DADN0EZPRYRWX_2.png  
  inflating: /../content/kaggle/carSide/1SEKLZ59C64QAASDJGFR_4.png  
  inflating: /../content/kaggle/carSide/WMCGONUMK9LT0GGQTYAB_8.png  
  inflating: /../content/kaggle/carSide/LUEJYD5RPH59REZF7EVH_0.png  
  inflating: /../content/kaggle/carSide/3746_jpg.rf.7b2bf6101a0b6d65fda37f25f663ad53_0.png  
 extracting: /../content/kaggle/carSide/9NATON8KR8MXS41QIU9T_4.png  
  inflating: /../content/kaggle/carSide/0BWTVBE3TKNH6ZEOTAMT_5.png  
  inflating: /../content/kaggle/carSide/H0KCYNL2WEEQCDMB5YS6_2.png  
  inflating: /../content/kaggle/carSide/V3AFVMUU50YQBUPUPWO3_0.png  
 extracting: /../con

In [ ]:
!mv /content/kaggle /content/train

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/processado/object-classification-challenge-2025.zip' '/content/object-classification-challenge-2025.zip'

In [ ]:
!unzip "/content/object-classification-challenge-2025.zip"  -d /content/

Archive:  /content/object-classification-challenge-2025.zip
  inflating: /content/objects/extra_file/sample_submission.csv  
  inflating: /content/objects/teste/1.png  
  inflating: /content/objects/teste/10.png  
  inflating: /content/objects/teste/100.png  
  inflating: /content/objects/teste/1000.png  
  inflating: /content/objects/teste/1001.png  
  inflating: /content/objects/teste/1002.png  
  inflating: /content/objects/teste/1003.png  
  inflating: /content/objects/teste/1004.png  
  inflating: /content/objects/teste/1005.png  
  inflating: /content/objects/teste/1006.png  
  inflating: /content/objects/teste/1007.png  
  inflating: /content/objects/teste/1008.png  
  inflating: /content/objects/teste/1009.png  
  inflating: /content/objects/teste/101.png  
  inflating: /content/objects/teste/1010.png  
  inflating: /content/objects/teste/1011.png  
  inflating: /content/objects/teste/1012.png  
  inflating: /content/objects/teste/1013.png  
  inflating: /content/objects/teste/

In [ ]:
"""
====================================================================
CNN SIMPLES COM ROBUSTEZ PARA GRAYSCALE
Arquitetura tradicional com camadas convolucionais sequenciais
====================================================================
"""

import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import json
import time
from collections import Counter
import zipfile
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFilter

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# ==================================================================
# 1. CONFIGURAÇÕES GLOBAIS
# ==================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Usando device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memória disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

CONFIG = {
    'num_classes': 6,
    'class_names': ["motorbike", "person", "bicycle", "car", "carSide", "carRear"],
    'class_to_id': {"motorbike": 1, "person": 2, "bicycle": 3, "car": 4, "carSide": 5, "carRear": 6},
    'img_size': 112,
    'batch_size': 64,
    'num_epochs': 80,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'dropout_rate': 0.5,
    'patience': 15,
    'seed': 42,
    'num_workers': 2,
    'device': device,
    'mixed_precision': True,

    # Parâmetros de augmentation
    'use_mixup': True,
    'mixup_alpha': 0.2,
    'label_smoothing': 0.1,
    'grayscale_prob': 0.2,  # 20% das imagens em treino viram grayscale

    # Caminhos
    'train_dir': '/content/train',
    'test_dir': '/content/objects/teste',
    'output_dir': '/content/objects/extra_file',
    'model_path': '/content/cnn_simple_grayscale_model.pth',

    'force_retrain': False
}

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])

# ==================================================================
# 2. AUGMENTATIONS CUSTOMIZADAS
# ==================================================================

class GaussianBlur:
    """Aplica Gaussian Blur para simular imagens borradas"""
    def __init__(self, p=0.5, radius_range=(0.5, 1.5)):
        self.p = p
        self.radius_range = radius_range

    def __call__(self, img):
        if np.random.random() < self.p:
            radius = np.random.uniform(*self.radius_range)
            return img.filter(ImageFilter.GaussianBlur(radius=radius))
        return img


class RandomGrayscale:
    """Converte para grayscale com probabilidade p"""
    def __init__(self, p=0.2):
        self.p = p

    def __call__(self, img):
        if np.random.random() < self.p:
            # Converte para grayscale e depois de volta para RGB (3 canais iguais)
            return img.convert('L').convert('RGB')
        return img


# ==================================================================
# 3. DICE LOSS COM LABEL SMOOTHING
# ==================================================================

class DiceLoss(nn.Module):
    """Dice Loss para otimização do F1-Score"""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        probs = F.softmax(logits, dim=1)

        if targets.dim() == 1:
            targets_one_hot = F.one_hot(targets, num_classes).float()
        else:
            targets_one_hot = targets

        intersection = (probs * targets_one_hot).sum(dim=0)
        cardinality = (probs + targets_one_hot).sum(dim=0)
        dice_score = (2. * intersection + self.smooth) / (cardinality + self.smooth)

        return 1 - dice_score.mean()


class CombinedLoss(nn.Module):
    """Combina Dice Loss com Cross Entropy (com label smoothing)"""
    def __init__(self, dice_weight=0.5, ce_weight=0.5, smooth=1.0, label_smoothing=0.1):
        super().__init__()
        self.dice_loss = DiceLoss(smooth=smooth)
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight

    def forward(self, logits, targets):
        # Se targets já é soft (mixup), usa KL divergence
        if targets.dim() > 1:
            dice = self.dice_loss(logits, targets)
            ce = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
        else:
            dice = self.dice_loss(logits, targets)
            ce = self.ce_loss(logits, targets)

        return self.dice_weight * dice + self.ce_weight * ce


# ==================================================================
# 4. MIXUP
# ==================================================================

def mixup_data(x, y, alpha=0.2):
    """Aplica MixUp data augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]

    # Retorna labels suaves para mixup
    y_mixed = lam * F.one_hot(y_a, CONFIG['num_classes']).float() + \
              (1 - lam) * F.one_hot(y_b, CONFIG['num_classes']).float()

    return mixed_x, y_mixed


# ==================================================================
# 5. ARQUITETURA CNN SIMPLES TRADICIONAL
# ==================================================================

class SimpleCNN(nn.Module):
    """CNN Simples com camadas convolucionais tradicionais"""
    def __init__(self, num_classes=6, dropout_rate=0.5):
        super().__init__()

        # Bloco Convolucional 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)  # 112 -> 56

        # Bloco Convolucional 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)  # 56 -> 28

        # Bloco Convolucional 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)  # 28 -> 14

        # Bloco Convolucional 4
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(2, 2)  # 14 -> 7

        # Bloco Convolucional 5
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        self.pool5 = nn.MaxPool2d(2, 2)  # 7 -> 3

        # Camadas Fully Connected
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(512 * 3 * 3, 1024)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(1024, 512)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(512, num_classes)

        self._initialize_weights()

    def _initialize_weights(self):
        """Inicialização de pesos usando He initialization"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Bloco 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)

        # Bloco 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)

        # Bloco 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)

        # Bloco 4
        x = self.conv4(x)
        x = self.bn4(x)
        x = F.relu(x)
        x = self.pool4(x)

        # Bloco 5
        x = self.conv5(x)
        x = self.bn5(x)
        x = F.relu(x)
        x = self.pool5(x)

        # Fully Connected
        x = self.flatten(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = F.relu(x)
        x = self.dropout2(x)

        x = self.fc3(x)

        return x


# ==================================================================
# 6. DATASET COM AUGMENTATIONS ROBUSTAS
# ==================================================================

class VehiclePersonDataset(Dataset):
    """Dataset COM augmentations robustas"""
    def __init__(self, image_paths, labels, transform=None, is_train=False):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if image.size != (112, 112):
                image = image.resize((112, 112), Image.BILINEAR)
        except Exception as e:
            print(f"Erro ao carregar {img_path}: {e}")
            image = Image.new('RGB', (112, 112), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        label = self.labels[idx]
        return image, label


class TestDataset(Dataset):
    """Dataset de teste simples (sem TTA)"""
    def __init__(self, image_paths):
        self.image_paths = image_paths

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        try:
            image = Image.open(img_path).convert('RGB')
            if image.size != (112, 112):
                image = image.resize((112, 112), Image.BILINEAR)
        except Exception as e:
            print(f"Erro ao carregar {img_path}: {e}")
            image = Image.new('RGB', (112, 112), (0, 0, 0))

        image = self.transform(image)

        return image, str(img_path)


def get_transforms(train=False):
    """Retorna transformações COM augmentations robustas"""
    if train:
        return transforms.Compose([
            RandomGrayscale(p=CONFIG['grayscale_prob']),  # 20% grayscale
            GaussianBlur(p=0.3, radius_range=(0.5, 1.5)),  # 30% blur
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])


# ==================================================================
# 7. TREINAMENTO COM MIXUP E ROBUSTEZ
# ==================================================================

def train_one_epoch(model, loader, criterion, optimizer, device, scaler, use_mixup=True):
    """Treina o modelo por uma época com mixup"""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_targets = []

    pbar = tqdm(loader, desc='Training', leave=False)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)

        # Aplica mixup
        if use_mixup and np.random.random() < 0.5:
            inputs, targets_mixed = mixup_data(inputs, targets, CONFIG['mixup_alpha'])
        else:
            targets_mixed = targets

        optimizer.zero_grad()

        if scaler and CONFIG['mixed_precision']:
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, targets_mixed)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, targets_mixed)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)

        # Para métricas, usa targets originais
        if targets_mixed.dim() > 1:
            targets_for_metrics = targets
        else:
            targets_for_metrics = targets_mixed

        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets_for_metrics.cpu().numpy())

        f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
        pbar.set_postfix({'loss': f'{running_loss/(pbar.n+1):.4f}', 'f1': f'{f1:.4f}'})

    final_f1 = f1_score(all_targets, all_preds, average='macro')
    return running_loss / len(loader), final_f1


def validate(model, loader, criterion, device):
    """Valida o modelo sem TTA"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in tqdm(loader, desc='Validation', leave=False):
            inputs, targets = inputs.to(device), targets.to(device)

            if inputs.size(0) == 1:
                continue

            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item()

            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    loss = running_loss / len(loader) if len(loader) > 0 else 0.0
    f1 = f1_score(all_targets, all_preds, average='macro') if len(all_targets) > 0 else 0.0

    return loss, f1, np.array(all_preds), np.array(all_targets)


def load_training_data(root_dir):
    """Carrega 100% dos dados de treinamento"""
    image_paths = []
    labels = []

    class_to_idx = {name: idx for idx, name in enumerate(CONFIG['class_names'])}

    for class_name, class_idx in class_to_idx.items():
        class_dir = Path(root_dir) / class_name
        if class_dir.exists():
            imgs = list(class_dir.glob('*.png'))
            image_paths.extend(imgs)
            labels.extend([class_idx] * len(imgs))
            print(f"  {class_name}: {len(imgs)} imagens")

    return np.array(image_paths), np.array(labels)


def train_full_model():
    """Treina o modelo com augmentations robustas"""
    print("\n" + "="*70)
    print("TREINAMENTO: CNN SIMPLES + Dice Loss + ROBUSTEZ GRAYSCALE/BLUR")
    print("="*70)

    all_paths, all_labels = load_training_data(CONFIG['train_dir'])
    print(f"\n✓ Total: {len(all_paths)} imagens carregadas")
    print(f"✓ Augmentations ativadas: Grayscale ({CONFIG['grayscale_prob']*100}%), Blur, MixUp")

    from sklearn.model_selection import train_test_split
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        all_paths, all_labels, test_size=0.2, stratify=all_labels,
        random_state=CONFIG['seed']
    )

    print(f"✓ Treino: {len(train_paths)} imagens (80%)")
    print(f"✓ Validação: {len(val_paths)} imagens (20%)")

    train_dataset = VehiclePersonDataset(train_paths, train_labels,
                                         get_transforms(True), is_train=True)
    val_dataset = VehiclePersonDataset(val_paths, val_labels,
                                       get_transforms(False), is_train=False)

    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG['batch_size'],
        shuffle=True, num_workers=CONFIG['num_workers'],
        pin_memory=True, drop_last=True
    )

    val_loader = DataLoader(
        val_dataset, batch_size=CONFIG['batch_size'],
        shuffle=False, num_workers=CONFIG['num_workers'],
        pin_memory=True, drop_last=False
    )

    model = SimpleCNN(CONFIG['num_classes'], CONFIG['dropout_rate']).to(device)

    criterion = CombinedLoss(
        dice_weight=0.6, ce_weight=0.4,
        label_smoothing=CONFIG['label_smoothing']
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2
    )
    scaler = torch.cuda.amp.GradScaler() if CONFIG['mixed_precision'] else None

    best_val_f1 = 0.0
    patience_counter = 0

    for epoch in range(1, CONFIG['num_epochs'] + 1):
        print(f"\n{'='*60}\nEpoch {epoch}/{CONFIG['num_epochs']}\n{'='*60}")

        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion,
            optimizer, device, scaler,
            CONFIG['use_mixup']
        )
        val_loss, val_f1, val_preds, val_targets = validate(
            model, val_loader, criterion, device
        )

        scheduler.step()

        print(f"Train - Loss: {train_loss:.4f}, F1-Score: {train_f1:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, F1-Score: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), CONFIG['model_path'])
            print(f"✓ Modelo salvo! (Best Val F1: {best_val_f1:.4f})")
        else:
            patience_counter += 1
            print(f"⚠ F1 não melhorou. Patience: {patience_counter}/{CONFIG['patience']}")

            if patience_counter >= CONFIG['patience']:
                print(f"\n⚠ Early stopping ativado em epoch {epoch}")
                print(f"   Melhor F1-Score: {best_val_f1:.4f}")
                break

        if epoch % 10 == 0:
            backup_path = CONFIG['model_path'].replace('.pth', f'_epoch{epoch}.pth')
            torch.save(model.state_dict(), backup_path)
            print(f"✓ Backup salvo: {backup_path}")

    print(f"\n✓ Treinamento concluído!")
    print(f"   Melhor F1-Score: {best_val_f1:.4f}")
    print(f"   Modelo final em: {CONFIG['model_path']}")

    return model


# ==================================================================
# 8. INFERÊNCIA SEM TTA
# ==================================================================

def classify_test_images(model):
    """Classifica imagens de teste sem TTA"""
    print("\n" + "="*70)
    print("CLASSIFICAÇÃO DE TESTE (SEM TTA)")
    print("="*70)

    test_dir = Path(CONFIG['test_dir'])
    test_images = sorted(list(test_dir.glob('*.png')))

    if len(test_images) == 0:
        print(f"❌ Nenhuma imagem encontrada em {test_dir}")
        return

    print(f"✓ {len(test_images)} imagens encontradas")

    test_dataset = TestDataset(test_images)
    test_loader = DataLoader(
        test_dataset, batch_size=32,
        shuffle=False, num_workers=CONFIG['num_workers']
    )

    model.eval()
    predictions = []
    filenames = []

    with torch.no_grad():
        for images, paths in tqdm(test_loader, desc='Classificando'):
            images = images.to(device)

            outputs = model(images)
            _, preds = outputs.max(1)

            predictions.extend(preds.cpu().numpy())
            filenames.extend([Path(p).name for p in paths])

    idx_to_id = {
        idx: CONFIG['class_to_id'][name]
        for idx, name in enumerate(CONFIG['class_names'])
    }
    category_ids = [idx_to_id[pred] for pred in predictions]

    results_df = pd.DataFrame({
        'Id': filenames,
        'Category': category_ids
    })

    results_df['sort_key'] = results_df['Id'].str.extract(r'(\d+)').astype(int)
    results_df = results_df.sort_values('sort_key').drop('sort_key', axis=1)

    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    output_path = Path(CONFIG['output_dir']) / 'submission.csv'
    results_df.to_csv(output_path, index=False)

    print(f"\n✓ Arquivo salvo em: {output_path}")
    print(f"\nResultado completo ({len(results_df)} linhas):")
    print(results_df.to_string(index=False))

    print("\nDistribuição das previsões:")
    for class_name in CONFIG['class_names']:
        class_id = CONFIG['class_to_id'][class_name]
        count = (results_df['Category'] == class_id).sum()
        print(f"  {class_name} (ID {class_id}): {count} imagens")

    return


# ==================================================================
# 9. EXECUÇÃO PRINCIPAL
# ==================================================================

def main():
    """Função principal"""

    model_exists = os.path.exists(CONFIG['model_path'])
    model_valid = False

    if model_exists and not CONFIG['force_retrain']:
        print(f"\n✓ Modelo encontrado: {CONFIG['model_path']}")

        try:
            print("  Verificando integridade do modelo...")
            model = SimpleCNN(CONFIG['num_classes'], CONFIG['dropout_rate']).to(device)
            model.load_state_dict(torch.load(CONFIG['model_path'], map_location=device))
            print("✓ Modelo carregado com sucesso!")
            model_valid = True
        except (RuntimeError, EOFError, zipfile.BadZipFile) as e:
            print(f"❌ Erro ao carregar modelo: {e}")
            print("⚠ O arquivo do modelo está corrompido!")
            print("  Deletando arquivo corrompido e iniciando novo treinamento...\n")
            try:
                os.remove(CONFIG['model_path'])
            except:
                pass
            model_valid = False

    if not model_valid:
        if CONFIG['force_retrain']:
            print("\n⚠ Flag 'force_retrain' ativada. Retreinando modelo...")
        elif not model_exists:
            print(f"\n⚠ Modelo não encontrado em {CONFIG['model_path']}")
            print("  Iniciando novo treinamento...")

        model = train_full_model()

    _ = classify_test_images(model)

    print("\n" + "="*70)
    print("✓ PROCESSO CONCLUÍDO COM SUCESSO!")
    print("="*70)


if __name__ == "__main__":
    main()

🚀 Usando device: cuda
   GPU: NVIDIA A100-SXM4-40GB
   Memória disponível: 42.47 GB

⚠ Modelo não encontrado em /content/cnn_simple_grayscale_model.pth
  Iniciando novo treinamento...

TREINAMENTO: CNN SIMPLES + Dice Loss + ROBUSTEZ GRAYSCALE/BLUR
  motorbike: 9999 imagens
  person: 10000 imagens
  bicycle: 10000 imagens
  car: 10000 imagens
  carSide: 10000 imagens
  carRear: 10000 imagens

✓ Total: 59999 imagens carregadas
✓ Augmentations ativadas: Grayscale (20.0%), Blur, MixUp
✓ Treino: 47999 imagens (80%)
✓ Validação: 12000 imagens (20%)

Epoch 1/80


Training:   0%|          | 0/749 [00:00<?, ?it/s]

Validation:   0%|          | 0/188 [00:00<?, ?it/s]

Train - Loss: 0.7202, F1-Score: 0.5539
Val   - Loss: 0.5031, F1-Score: 0.8251
✓ Modelo salvo! (Best Val F1: 0.8251)

Epoch 2/80


Training:   0%|          | 0/749 [00:00<?, ?it/s]

...

Training:   0%|          | 0/749 [00:00<?, ?it/s]

Validation:   0%|          | 0/188 [00:00<?, ?it/s]

Train - Loss: 0.2921, F1-Score: 0.7545
Val   - Loss: 0.2831, F1-Score: 0.9525
⚠ F1 não melhorou. Patience: 12/15
✓ Backup salvo: /content/cnn_simple_grayscale_model_epoch80.pth

✓ Treinamento concluído!
   Melhor F1-Score: 0.9579
   Modelo final em: /content/cnn_simple_grayscale_model.pth

CLASSIFICAÇÃO DE TESTE (SEM TTA)
✓ 2682 imagens encontradas


Classificando:   0%|          | 0/84 [00:00<?, ?it/s]


✓ Arquivo salvo em: /content/objects/extra_file/submission.csv

Resultado completo (2682 linhas):
      Id  Category
   1.png         5
   2.png         3
   3.png         5
   4.png         6
   5.png         4
   6.png         3
   7.png         2
   8.png         2
   9.png         5
  10.png         5
  11.png         4
  12.png         4
  13.png         4
  14.png         5
  15.png         3
  16.png         3
  17.png         5
  18.png         5
  19.png         5
  20.png         3
  21.png         2
  22.png         4
  23.png         3
  24.png         2
  25.png         3
  26.png         3
  27.png         3
  28.png         2
  29.png         2
  30.png         2
  31.png         2
  32.png         2
  33.png         4
  34.png         5
  35.png         5
  36.png         2
  37.png         2
  38.png         2
  39.png         3
  40.png         2
  41.png         2
  42.png         2
  43.png         2
  44.png         2
  45.png         2
  46.png         2
  47.png

In [ ]:
!cp '/content/cnn_simple_grayscale_model.pth' '/content/drive/MyDrive/IC009/dataset/processado/cnn_simple_grayscale_model.pth'

In [ ]:
!cp '/content/objects/extra_file/submission.csv' '/content/drive/MyDrive/IC009/dataset/processado/submission.csv'